In [ ]:
import os
import json
import torch
import pandas as pd
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import classification_report, confusion_matrix

ROIS_ROOT    = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET\roi_Output1"   # ROI images folder
LABELS_CSV   = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET\roi_labels1.csv"
MODEL_PATH   = r"C:\vs code\PCB_circuit\PCB_circuit_output\pcb_model.pth"
CLASS_JSON   = r"C:\vs code\PCB_circuit\PCB_circuit_output\class_to_idx.json"
OUT_DIR      = r"C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET"   # folder for annotated images

os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
BATCH_SIZE = 4
NUM_WORKERS = 0
PIN_MEMORY = False
print(f"Using device: {DEVICE}")

Using device: cuda


In [45]:
# Load class mapping
with open(CLASS_JSON, "r") as f:
    class_to_idx = json.load(f)
idx_to_class = {v:k for k,v in class_to_idx.items()}

# Load CSV
df = pd.read_csv(LABELS_CSV)
df = df.rename(columns={"file_name": "filename"})
df["abs_path"] = df.apply(lambda row: os.path.join(ROIS_ROOT, row["label"], row["filename"]), axis=1)

# Check missing files
missing = df[~df["abs_path"].apply(os.path.exists)]
if not missing.empty:
    print("⚠ Missing files detected:\n", missing)
else:
    print("✅ All files exist.")

# For testing, use all data or split as needed
test_df = df.copy()

# Dataset & DataLoader
test_tfms = build_transforms()
test_ds = ROIDataset(test_df, transform=test_tfms, class_to_idx=class_to_idx)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


✅ All files exist.


In [46]:
import timm
def build_model(num_classes):
    return timm.create_model("efficientnet_b4", pretrained=False, num_classes=num_classes)

model = build_model(num_classes=len(class_to_idx))
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print("✅ Model loaded and ready for testing")


C:\Users\manne\AppData\Local\Temp\ipykernel_12784\3461123550.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DE

✅ Model loaded and ready for testing


In [47]:
os.makedirs(OUT_DIR, exist_ok=True)

all_labels, all_preds = [], []

font = ImageFont.load_default()  # Simple font for annotations

with torch.no_grad():
    for images, labels, filenames in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

        # Annotate & save images
        images = images.cpu()
        for i in range(len(filenames)):
            img_tensor = images[i]
            # Convert tensor to PIL image
            img = transforms.ToPILImage()(img_tensor)
            draw = ImageDraw.Draw(img)
            pred_class = idx_to_class[int(preds[i])]
            true_class = idx_to_class[int(labels[i])]
            text = f"P: {pred_class} | T: {true_class}"
            color = "green" if pred_class == true_class else "red"
            draw.text((5,5), text, fill=color, font=font)
            img.save(os.path.join(OUT_DIR, filenames[i]))

print(f"✅ Annotated images saved in {OUT_DIR}")


✅ Annotated images saved in C:\vs code\PCB_circuit\PCB_DATASET\PCB_DATASET


In [48]:
print("Classification Report:")
print(classification_report(all_labels, all_preds,
                            target_names=[idx_to_class[i] for i in sorted(idx_to_class)]))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


Classification Report:
                 precision    recall  f1-score   support

   Missing_hole       1.00      1.00      1.00       497
     Mouse_bite       0.94      0.93      0.94       191
   Open_circuit       0.95      0.92      0.93       168
          Short       0.90      0.96      0.93       169
           Spur       0.95      0.88      0.91       857
Spurious_copper       0.92      0.97      0.95      1261

       accuracy                           0.94      3143
      macro avg       0.94      0.94      0.94      3143
   weighted avg       0.94      0.94      0.94      3143

Confusion Matrix:
[[ 495    0    0    0    1    1]
 [   0  178    8    0    4    1]
 [   0   11  155    0    2    0]
 [   0    0    0  162    0    7]
 [   0    0    1   11  753   92]
 [   0    0    0    7   33 1221]]
